<a href="https://colab.research.google.com/github/cyruslayo/nba_bot/blob/main/notebooks/nba_win_prob_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Win Probability — Colab Training Notebook

Trains an XGBoost win-probability model using pre-built PBP CSVs from `nba-on-court`.
Saves artifacts to Google Drive for use with `nba-bot-scan`.

**Run cells top-to-bottom on a fresh runtime.**

In [1]:
# Cell 1 — Setup: install dependencies and nba_bot package
# Step 1: Install data loader
!pip install "nba-on-court>=0.2.0,<1.0" -q
# Step 2: Force upgrade numpy and install training tools
!pip install "numpy>=2.0" xgboost scikit-learn pandas joblib tqdm nba_api --upgrade -q

import os
if not os.path.exists('/content/nba_bot'):
    print("Cloning repository...")
    !git clone https://github.com/cyruslayo/nba_bot.git /content/nba_bot
else:
    print("Repository exists. Pulling latest changes...")
    !git -C /content/nba_bot pull

# Step 3: Install the cloned repo in editable mode to fix internal imports
!pip install -e /content/nba_bot -q

print('✅ Setup complete (Package installed in editable mode)')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nba-bot 0.1.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, bu

In [2]:
# Cell 2 — Mount Google Drive (model artifacts will be saved here)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/nba_bot/'
import os; os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output dir: {DRIVE_OUTPUT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/nba_bot/


In [3]:
# Cell 3 — Config: choose seasons and feature tier
SEASONS = [2021, 2022, 2023, 2024]  # season start years
USE_ADVANCED_FEATURES = True         # True = Tier-2 model (better), False = Tier-1 (faster)

print(f'Seasons: {SEASONS}')
print(f'Tier: {"T2 (advanced)" if USE_ADVANCED_FEATURES else "T1 (baseline)"}')

Seasons: [2021, 2022, 2023, 2024]
Tier: T2 (advanced)


In [4]:
# Cell 4 — Download play-by-play data (with manual fallback)
import nba_on_court as noc
import pandas as pd
import os

def fix_columns(df):
    """Normalize GAME_ID column — applied per file before concat."""
    if df is None: return None
    df.columns = [c.upper() for c in df.columns]
    if 'GAME_ID' not in df.columns:
        possible_id_cols = [c for c in df.columns if 'NBASTATS' in c or 'GAMEID' in c or 'GAME_ID' in c]
        if possible_id_cols:
            df = df.rename(columns={possible_id_cols[0]: 'GAME_ID'})
    return df

def load_data_safe(seasons, data_type):
    try:
        df = noc.load_nba_data(seasons=seasons, data=data_type)
        if df is not None and len(df) > 0:
            return fix_columns(df)
    except Exception as e:
        print(f'Library load failed for {data_type}: {e}')

    # Fallback: fix columns on EACH file before concat so GAME_ID is consistent
    dfs = []
    for s in seasons:
        fpath = f'/content/{data_type}_{s}.tar.xz'
        if os.path.exists(fpath):
            print(f'Manually loading {fpath}...')
            raw = pd.read_csv(fpath, compression='xz')
            fixed = fix_columns(raw)
            if 'GAME_ID' in fixed.columns:
                print(f'  → {s}: {len(fixed):,} rows, {fixed["GAME_ID"].nunique():,} games')
            dfs.append(fixed)
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_nbastats = load_data_safe(SEASONS, 'nbastats')
if df_nbastats is None:
    raise ValueError("Could not load nbastats. Ensure files exist in /content/")

print(f'nbastats: {len(df_nbastats):,} rows, {df_nbastats["GAME_ID"].nunique():,} unique games')

df_pbpstats = None
if USE_ADVANCED_FEATURES:
    df_pbpstats = load_data_safe(SEASONS, 'pbpstats')
    if df_pbpstats is not None:
        print(f'pbpstats: {len(df_pbpstats):,} rows, {df_pbpstats["GAME_ID"].nunique():,} unique games')
    else:
        print('Warning: pbpstats not found/loaded')

assert len(df_nbastats) > 100_000, f'Expected >100k rows, got {len(df_nbastats)}'
print('✅ Data load OK')


Manually loading /content/nbastats_2021.tar.xz...
  → 2021: 570,364 rows, 1,230 games
Manually loading /content/nbastats_2022.tar.xz...
  → 2022: 574,411 rows, 1,230 games
Manually loading /content/nbastats_2023.tar.xz...
  → 2023: 567,666 rows, 1,230 games
Manually loading /content/nbastats_2024.tar.xz...
  → 2024: 574,361 rows, 1,230 games
nbastats: 2,286,802 rows, 4,920 unique games
Manually loading /content/pbpstats_2021.tar.xz...
  → 2021: 482,409 rows, 1,230 games
Manually loading /content/pbpstats_2022.tar.xz...
  → 2022: 486,515 rows, 1,230 games
Manually loading /content/pbpstats_2023.tar.xz...
  → 2023: 478,626 rows, 1,228 games
Manually loading /content/pbpstats_2024.tar.xz...
  → 2024: 483,127 rows, 1,230 games
pbpstats: 1,930,677 rows, 4,918 unique games
✅ Data load OK


In [5]:
import sys
import os
import importlib.util

# Patch the source file to fix the nba_api compatibility issue
!sed -i "s/per_mode_simple/per_mode_detailed/g" /content/nba_bot/nba_bot/team_stats_cache.py

player_ratings = {}
if USE_ADVANCED_FEATURES:
    # 1. Manually load 'nba_bot.config' into sys.modules first
    config_path = "/content/nba_bot/nba_bot/config.py"
    config_spec = importlib.util.spec_from_file_location("nba_bot.config", config_path)
    config_mod = importlib.util.module_from_spec(config_spec)
    sys.modules["nba_bot.config"] = config_mod
    config_spec.loader.exec_module(config_mod)

    # 2. Now manually load 'team_stats_cache.py'
    module_file = "/content/nba_bot/nba_bot/team_stats_cache.py"
    spec = importlib.util.spec_from_file_location("nba_bot.team_stats_cache", module_file)
    tsc_module = importlib.util.module_from_spec(spec)
    sys.modules["nba_bot.team_stats_cache"] = tsc_module
    spec.loader.exec_module(tsc_module)

    # 3. Access the function
    refresh_team_stats = tsc_module.refresh_team_stats
    player_ratings = refresh_team_stats(output_path='/content/team_stats.json')
    print(f'Loaded ratings for {len(player_ratings)} teams')
    assert len(player_ratings) > 0, 'No team ratings returned — check NBA API'
else:
    print('Skipping team ratings (Tier-1 mode)')

Loaded ratings for 30 teams


In [6]:
# Cell 6 — Feature engineering
import importlib.util
import sys
import pandas as pd

# Directly load the optimized features.py from disk
features_path = "/content/nba_bot/nba_bot/features.py"
spec = importlib.util.spec_from_file_location("nba_bot.features", features_path)
features_module = importlib.util.module_from_spec(spec)
sys.modules["nba_bot.features"] = features_module
spec.loader.exec_module(features_module)

build_game_state_rows = features_module.build_game_state_rows

# Improved column normalization to ensure GAME_ID exists
def fix_columns(df):
    if df is None: return None
    df.columns = [c.upper() for c in df.columns]
    # Look for common ID patterns if GAME_ID is missing
    if 'GAME_ID' not in df.columns:
        possible_id_cols = [c for c in df.columns if 'NBASTATS' in c or 'GAMEID' in c or 'GAME_ID' in c]
        if possible_id_cols:
            df = df.rename(columns={possible_id_cols[0]: 'GAME_ID'})
    return df

# ── Diagnostics (run before build_game_state_rows) ──────────────────────────
print(f"Total rows:       {len(df_nbastats):,}")
print(f"Unique GAME_IDs:  {df_nbastats['GAME_ID'].nunique():,}")
print(f"Null GAME_IDs:    {df_nbastats['GAME_ID'].isna().sum():,}")
print()
print("Sample GAME_IDs:")
print(df_nbastats['GAME_ID'].dropna().unique()[:10])
print()
print("Games per apparent season (first 4 digits of ID):")
df_nbastats['season_prefix'] = df_nbastats['GAME_ID'].astype(str).str[:4]
print(df_nbastats.groupby('season_prefix')['GAME_ID'].nunique())


print("Starting feature engineering (this should only take a few minutes)...")

# Running optimized build_game_state_rows natively
df = build_game_state_rows(
    df_nbastats    = df_nbastats,
    df_pbpstats    = df_pbpstats,
    player_ratings = player_ratings,
    use_advanced   = USE_ADVANCED_FEATURES,
)

feature_tier = 'T2' if USE_ADVANCED_FEATURES else 'T1'
print(f'\nDataset shape:   {df.shape}')
print(f'Class balance:   {df["home_win"].mean():.1%}  (should be ~50%)')
print(f'Feature tier:    {feature_tier}')
print(f'Games:           {df["game_id"].nunique():,}')
print()
print(df.head(3))


Total rows:       2,286,802
Unique GAME_IDs:  4,920
Null GAME_IDs:    4

Sample GAME_IDs:
[22100001. 22100010. 22100100. 22101000. 22101001. 22101002. 22101003.
 22101004. 22101005. 22101006.]

Games per apparent season (first 4 digits of ID):
season_prefix
2210    1230
2220    1230
2230    1230
2240    1230
Name: GAME_ID, dtype: int64
Starting feature engineering (this should only take a few minutes)...

Dataset shape:   (2286798, 11)
Class balance:   55.3%  (should be ~50%)
Feature tier:    T2
Games:           4,920

   score_diff  time_remaining  time_pressure  game_progress  period  \
0         1.0          2742.0       0.019094       0.047917       1   
1         1.0          2732.0       0.019128       0.051389       1   
2         1.0          2730.0       0.019135       0.052083       1   

   is_overtime  starttype_encoded  player_quality_home  player_quality_away  \
0            0                  2                 -3.7                 -9.4   
1            0                  

In [7]:
# Cell 7 — Train models & evaluate
# Target: T1 AUC-ROC > 0.92 | T2 AUC-ROC > 0.93
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from nba_bot.config import FEATURES_T1, FEATURES_T2

feature_cols = FEATURES_T1 + (FEATURES_T2 if USE_ADVANCED_FEATURES else [])

df_train = df[df['time_remaining'] < 2870].dropna(subset=feature_cols + ['home_win'])
X = df_train[feature_cols].values
y = df_train['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# Logistic Regression baseline
lr = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
print(f'LR  → log_loss={log_loss(y_test, lr_probs):.4f}  brier={brier_score_loss(y_test, lr_probs):.4f}  auc={roc_auc_score(y_test, lr_probs):.4f}')

# XGBoost main model
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, verbosity=0,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_probs)
print(f'XGB → log_loss={log_loss(y_test, xgb_probs):.4f}  brier={brier_score_loss(y_test, xgb_probs):.4f}  auc={xgb_auc:.4f}')

# AC 15 validation
min_auc = 0.91 if USE_ADVANCED_FEATURES else 0.90
assert xgb_auc > min_auc, f'AUC-ROC {xgb_auc:.4f} below threshold {min_auc}'
print(f'✅ AUC-ROC check passed ({xgb_auc:.4f} > {min_auc})')

Train: 454,251  |  Test: 113,563
LR  → log_loss=0.4977  brier=0.1688  auc=0.8247
XGB → log_loss=0.4913  brier=0.1676  auc=0.8268


AssertionError: AUC-ROC 0.8268 below threshold 0.91

In [ ]:
# Cell 8 — Save model artifacts to Google Drive
import joblib

model_name = f"xgb_model_t{'2' if USE_ADVANCED_FEATURES else '1'}.pkl"

# Save XGBoost model
joblib.dump(xgb_model, f'{DRIVE_OUTPUT}{model_name}')

# Save feature_cols.pkl — records the ordered feature list used at training time.
# model.py's load_model() reads this to validate feature order at inference (F12 fix).
joblib.dump(feature_cols, f'{DRIVE_OUTPUT}feature_cols.pkl')

print(f'✅ Saved: {DRIVE_OUTPUT}{model_name}')
print(f'✅ Saved: {DRIVE_OUTPUT}feature_cols.pkl')
print()
print('Download these files from Drive, then:')
print(f'  export NBA_BOT_MODEL_PATH=/path/to/{model_name}')
print('  nba-bot-scan --mode test')

In [ ]:
# Cell 9 — Inference smoke test
# nba_bot is installed in Cell 1, so this import will succeed (F4 fix)
from nba_bot.features import compute_features

model_check = joblib.load(f'{DRIVE_OUTPUT}{model_name}')
X_test_smoke = compute_features(home_score=88, away_score=84, period=4, clock='3:20')
prob = float(model_check.predict_proba(X_test_smoke)[0][1])

assert 0.4 < prob < 0.99, f'Suspicious output: {prob}  (expected 0.4–0.99 for this scenario)'
print(f'✅ Inference OK — home win prob: {prob:.3f}')
print(f'   (Lakers leading by 4 with 3:20 left in Q4)')